In [3]:
import pandas as pd
import math

df = pd.read_csv("play_tennis.csv")

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [5]:
df.head()

,day,outlook,temp,humidity,wind,play
0,D1,Sunny,Hot,High,Weak,No
1,D2,Sunny,Hot,High,Strong,No
2,D3,Overcast,Hot,High,Weak,Yes
3,D4,Rain,Mild,High,Weak,Yes
4,D5,Rain,Cool,Normal,Weak,Yes


In [7]:
def entropy(data):
    labels = data.iloc[:, -1]
    total = len(labels)

    counts = labels.value_counts()

    ent = 0

    for count in counts:
        p = count / total
        ent -= p * math.log2(p)

    return ent



In [8]:
def information_gain(data, feature):
    total_entropy = entropy(data)

    values = data[feature].unique()

    weighted_entropy = 0

    for value in values:
        subset = data[data[feature] == value]
        weighted_entropy += (len(subset) / len(data)) * entropy(subset)

    gain = total_entropy - weighted_entropy

    return gain



In [9]:
def best_feature(data):
    features = data.columns[:-1]

    gains = {}

    for feature in features:
        gains[feature] = information_gain(data, feature)

    best = max(gains, key=gains.get)

    return best



In [10]:
def majority_class(data):
    return data.iloc[:, -1].mode()[0]

def build_tree(data):

    labels = data.iloc[:, -1]

    if len(labels.unique()) == 1:
        return labels.iloc[0]

    if len(data.columns) == 1:
        return majority_class(data)

    best = best_feature(data)

    tree = {best: {}}

    for value in data[best].unique():

        subset = data[data[best] == value]

        subset = subset.drop(columns=[best])

        tree[best][value] = build_tree(subset)

    return tree

decision_tree = build_tree(df)

print("Decision Tree:")
print(decision_tree)

Decision Tree:
{'day': {'D1': 'No', 'D2': 'No', 'D3': 'Yes', 'D4': 'Yes', 'D5': 'Yes', 'D6': 'No', 'D7': 'Yes', 'D8': 'No', 'D9': 'Yes', 'D10': 'Yes', 'D11': 'Yes', 'D12': 'Yes', 'D13': 'Yes', 'D14': 'No'}}


In [11]:
def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    feature = next(iter(tree))
    value = sample[feature]

    if value in tree[feature]:
        return predict(tree[feature][value], sample)
    else:
        return "Unknown"

In [12]:
train_data, test_data = train_test_split(
    df,
    test_size=0.3,
    random_state=42
)

In [13]:
tree = build_tree(train_data)

print("Decision Tree:")
print(tree)

Decision Tree:
{'day': {'D9': 'Yes', 'D3': 'Yes', 'D2': 'No', 'D14': 'No', 'D5': 'Yes', 'D8': 'No', 'D11': 'Yes', 'D4': 'Yes', 'D7': 'Yes'}}


In [14]:
train_predictions = []

for i in range(len(train_data)):
    sample = train_data.iloc[i]
    pred = predict(tree, sample)
    train_predictions.append(pred)

In [15]:
test_predictions = []

for i in range(len(test_data)):
    sample = test_data.iloc[i]
    pred = predict(tree, sample)
    test_predictions.append(pred)

train_actual = train_data.iloc[:, -1]
test_actual = test_data.iloc[:, -1]

train_acc = accuracy_score(train_actual, train_predictions)
test_acc = accuracy_score(test_actual, test_predictions)

print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)

Training Accuracy: 1.0
Testing Accuracy: 0.0


In [16]:
if train_acc > 0.90 and test_acc < 0.70:
    print("Result: Model is Overfitting")
elif train_acc < 0.70 and test_acc < 0.70:
    print("Result: Model is Underfitting")
else:
    print("Result: Model is Good ")

Result: Model is Overfitting
